# Set 04 – Vorverarbeitung mit scikit-learn

scikit-learn besteht nicht nur aus Modellen. Es bietet Transformer, die Daten nach festen Regeln vorbereiten. Hier werden die Bausteine ohne Klassifikation und ohne Vorhersage eingeführt.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 1. Beispieldaten

Die Referenzdaten enthalten Zahlen, Kategorien und fehlende Werte. Die zweite Tabelle simuliert später eintreffende Daten. München kommt nur dort vor und ist eine unbekannte Kategorie.

In [ ]:
referenz = pd.DataFrame({
    "alter": [24, 37, np.nan, 52, 46, 31],
    "einkommen": [3100, 4200, 3900, np.nan, 6100, 3600],
    "stadt": ["Berlin", "Köln", "Berlin", "Hamburg", np.nan, "Köln"],
})
neu = pd.DataFrame({
    "alter": [29, np.nan],
    "einkommen": [3500, 4700],
    "stadt": ["München", "Berlin"],
})
display(referenz)
display(neu)

## 2. Das Transformer-Prinzip

- fit(X) lernt benötigte Werte oder Kategorien aus X.
- transform(X) wendet die gelernten Regeln an.
- fit_transform(X) führt beides direkt nacheinander aus.

Neue Daten werden nur mit transform() verarbeitet. Sonst würden die Regeln verändert.

## 3. Fehlende Zahlen mit SimpleImputer

Der Imputer lernt für jede Zahlenspalte den Median. Das Attribut statistics_ zeigt die gelernten Ersatzwerte.

In [ ]:
zahlen = referenz[["alter", "einkommen"]]
zahlen_imputer = SimpleImputer(strategy="median")
zahlen_aufbereitet = zahlen_imputer.fit_transform(zahlen)
print("Gelernte Medianwerte:", zahlen_imputer.statistics_)
display(pd.DataFrame(zahlen_aufbereitet, columns=zahlen.columns))

## 4. Zahlen mit StandardScaler skalieren

Der Scaler lernt Mittelwerte und Standardabweichungen. Danach liegen die transformierten Spalten ungefähr um 0 und sind vergleichbar skaliert.

In [ ]:
scaler = StandardScaler()
zahlen_skaliert = scaler.fit_transform(zahlen_aufbereitet)
print("Gelernte Mittelwerte:", scaler.mean_.round(2))
print("Gelernte Skalierungen:", scaler.scale_.round(2))
display(pd.DataFrame(zahlen_skaliert, columns=zahlen.columns).round(2))

## 5. Fehlende Kategorien auffüllen

most_frequent ersetzt fehlende Werte durch die häufigste Ausprägung.

In [ ]:
kategorie = referenz[["stadt"]]
kategorie_imputer = SimpleImputer(strategy="most_frequent")
kategorie_aufbereitet = kategorie_imputer.fit_transform(kategorie)
print("Häufigste Kategorie:", kategorie_imputer.statistics_)
display(pd.DataFrame(kategorie_aufbereitet, columns=["stadt"]))

## 6. Kategorien mit OneHotEncoder codieren

Für jede bekannte Kategorie entsteht eine 0/1-Spalte. handle_unknown="ignore" verhindert einen Fehler bei unbekannten Kategorien.

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
stadt_codiert = encoder.fit_transform(kategorie_aufbereitet)
print("Gelernte Kategorien:", encoder.categories_)
display(pd.DataFrame(stadt_codiert, columns=encoder.get_feature_names_out(["stadt"])))

## 7. Gelernte Regeln auf neue Daten anwenden

Für neu verwenden wir ausschließlich transform(). Medianwerte, Skalierung und Kategorien stammen weiterhin aus der Referenztabelle.

In [ ]:
neu_zahlen = zahlen_imputer.transform(neu[["alter", "einkommen"]])
neu_zahlen = scaler.transform(neu_zahlen)
neu_stadt = kategorie_imputer.transform(neu[["stadt"]])
neu_stadt = encoder.transform(neu_stadt)

display(pd.DataFrame(neu_zahlen, columns=["alter", "einkommen"]).round(2))
display(pd.DataFrame(neu_stadt, columns=encoder.get_feature_names_out(["stadt"])))

## Zwischenfazit

Die Transformer funktionieren, aber Reihenfolge und Spalten wurden manuell verwaltet. Im nächsten Notebook bündeln Pipeline und ColumnTransformer diese Schritte reproduzierbar.